In [4]:
import numpy as np

# 假设你的文件名是'output_file.npz'
file_name = 'patent_D_embeddings_0.npz'

# 使用np.load()函数来读取npz文件
with np.load(file_name) as data:
    # 列出文件中所有数组的名称
    print(data.files)
    
    # 访问embeddings和ids数组
    embeddings_array = data['embeddings']
    ids_array = data['ids']
    
    # 打印数组或执行其他操作
    print("Embeddings Array:")
    print(embeddings_array)
    
    print("IDs Array:")
    print(ids_array)


['embeddings', 'ids']
Embeddings Array:
[[ 0.002491   -0.00783539 -0.05648804 ... -0.00778198  0.0060463
  -0.02053833]
 [ 0.01200104 -0.04959106  0.00019848 ... -0.02311707  0.00577545
  -0.04223633]
 [-0.03353882  0.03778076 -0.02146912 ... -0.0076561  -0.03607178
  -0.08044434]
 ...
 [ 0.00063181 -0.00747681 -0.06561279 ... -0.03771973 -0.01292419
  -0.05895996]
 [ 0.00093842 -0.00125694 -0.00811768 ...  0.00361443 -0.00070763
  -0.03161621]
 [ 0.02799988  0.00341797  0.02339172 ...  0.0322876   0.0044136
  -0.09960938]]
IDs Array:
[41947446 41950716 41950838 ... 38759163 38763730 38766056]


In [5]:
embeddings_array.shape

(341518, 512)

In [6]:
min(ids_array)

86

In [7]:
import pandas as pd
df = pd.read_csv("patent_data_D_cleaned.csv")
df.head()

,id,title,main_ipc,brief
0,41947446,风道系统、滚筒组件、干衣设备及控制方法,D06F58/20,风道系统、滚筒组件、干衣设备及控制方法，涉及干衣设备技术领域，解决了现有技术中存在的热泵干衣...
1,41950716,用于造纸机车速联锁系统的高压清洗结构,D06B1/02,用于造纸机车速联锁系统的高压清洗结构，包括高压喷淋管和摆动器，摆动器的输出端处设有螺杆，螺杆...
2,41950838,一种熔体一步法生产线的生产制作流程,D01F8/14,熔体一步法生产线的生产制作流程，具体流程步骤如下：S1、高粘半消光PET切片经过干燥组件系统...
3,41951009,一种便于卡接的柔性打包带结构加工用裁剪设备,D06H7/02,便于卡接的柔性打包带结构加工用裁剪设备，涉及裁剪设备技术领域，为解决现有的裁剪设备在使用的过...
4,41952143,一种球形结构晾衣架,D06F57/00,本实用新型提供一种球形结构晾衣架，涉及晾衣架技术领域，包括两个圆环框架，每个所述圆环框架的相...


In [8]:
df.tail()

,id,title,main_ipc,brief
341513,38747850,一种具有辅助画线结构的服装设计用画线定型装置,D06H1/04,具有辅助画线结构的服装设计用画线定型装置，涉及服装设计技术领域，为了解决由于布料有一定的韧性...
341514,38754872,一种用于缝制鞋套松紧带的缝纫机,D05B35/06,本实用新型属于缝纫机技术领域，特别涉及一种用于缝制鞋套松紧带的缝纫机。所述缝纫机包括缝纫机主...
341515,38759163,一种聚乙烯醇原液着色用水性色浆及其制备方法,D01F1/04,聚乙烯醇原液着色用水性色浆及其制备方法，属于水性色浆技术领域。本发明的水性色浆包括以下原料：...
341516,38763730,用于洗涤设备的控制方法,D06F33/32,家电技术领域，具体提供一种用于洗涤设备的控制方法，现有洗涤设备重新上电后直接运行掉电之前的操...
341517,38766056,一种钢丝绳编织机,D04C3/46,钢丝绳编织机技术领域，公开了一种钢丝绳编织机，包括工作台、驱动装置和编织小车，工作台上设有腰...


In [2]:
import cupy as cp
import time
cp.cuda.Device(2).use()
# 生成7,000,000 x 512的大矩阵
start_time = time.time()
A = cp.random.random((860000, 512)).astype(cp.float32)
print(f"生成大矩阵耗时: {time.time() - start_time:.2f}秒")
print(f"大矩阵形状: {A.shape}, 内存占用: {A.nbytes / (1024**3):.2f} GB")

# 生成10 x 512的查询矩阵
B = cp.random.random((100, 512)).astype(cp.float32)
print(f"查询矩阵形状: {B.shape}")

# 执行点积操作
start_time = time.time()
result = cp.dot(A, B.T)  # 结果将是7,000,000 x 10的矩阵
cp.cuda.Stream.null.synchronize()  # 确保计算完成
dot_time = time.time() - start_time

print(f"点积计算耗时: {dot_time:.2f}秒")
print(f"结果矩阵形状: {result.shape}")
print(f"结果矩阵前5行:\n{result[:5]}")

生成大矩阵耗时: 0.04秒
大矩阵形状: (860000, 512), 内存占用: 1.64 GB
查询矩阵形状: (100, 512)
点积计算耗时: 0.02秒
结果矩阵形状: (860000, 100)
结果矩阵前5行:
[[122.80902  123.57507  132.34059  126.63123  129.23048  134.31705
  127.54985  123.84535  136.2579   127.354416 131.97687  134.87029
  128.48079  130.48706  125.674225 132.39885  134.46066  136.16797
  122.36594  126.83964  132.5158   128.47568  130.79915  132.19623
  126.46248  131.71622  131.55435  137.37442  125.70055  136.51009
  125.99857  128.62599  132.14691  126.771736 131.71503  129.27266
  128.70064  127.5236   132.1939   130.21187  127.523254 125.27825
  128.55707  133.56512  127.395836 128.02313  132.21767  127.55567
  129.48178  136.5259   134.32487  126.48962  133.17906  134.32764
  135.17027  128.27347  130.48558  131.73178  119.42966  128.09142
  133.7079   133.21797  128.23029  133.41382  130.74657  125.198784
  137.29451  124.82651  128.54187  127.35378  130.56992  131.25832
  129.7043   129.18121  126.3085   128.00475  130.08452  134.28702
  124.20561  

In [10]:
del A, B

In [ ]:
# 生成7,000,000 x 512的大矩阵
start_time = time.time()
C = cp.random.random((500000, 512)).astype(cp.float32)
print(f"生成大矩阵耗时: {time.time() - start_time:.2f}秒")
print(f"大矩阵形状: {C.shape}, 内存占用: {A.nbytes / (1024**3):.2f} GB")

OutOfMemoryError: Out of memory allocating 28,672,000,000 bytes (allocated so far: 43,008,000,000 bytes).

In [1]:
import pandas as pd
import numpy as np
import os
import gc
import time
import cupy as cp
from tqdm import tqdm
from cupyx.scipy.sparse import csr_matrix as cp_csr_matrix
from scipy.sparse import csr_matrix, save_npz, vstack, load_npz

def setup_gpu(gpu_id=0):
    """设置并初始化GPU"""
    cp.cuda.Device(gpu_id).use()
    device_name = cp.cuda.runtime.getDeviceProperties(gpu_id)['name'].decode()
    print(f"使用GPU {gpu_id}: {device_name}")

def load_data(patent_file, ipc_file):
    """加载和预处理专利数据"""
    print("正在读取数据文件...")
    patent_df = pd.read_csv(patent_file)
    ipc_df = pd.read_csv(ipc_file)
    
    print("正在合并IPC类别信息...")
    merged_df = pd.merge(
        patent_df,
        ipc_df,
        left_on='main_ipc',
        right_on='level_code',
        how='inner'
    )
    
    # 只保留需要的列以减少内存使用
    needed_columns = ['id', 'title', 'brief', 'level1_code', 'level2_code', 
                       'level3_code', 'level4_code', 'level5_code']
    return merged_df[needed_columns]

def create_feature_matrices(merged_df, weights):
    """为每个IPC级别创建特征矩阵"""
    print("进行高效预处理...")
    n = len(merged_df)
    feature_matrices = {}
    feature_matrices_gpu = {}
    
    for level, weight in weights.items():
        print(f"处理 {level}...")
        # 创建编码字典
        unique_values = merged_df[level].dropna().unique()
        code_to_idx = {code: idx for idx, code in enumerate(unique_values)}
        
        # 为每个专利创建行索引和列索引
        rows, cols, data = [], [], []
        
        for i, code in enumerate(merged_df[level]):
            if pd.notna(code):
                rows.append(i)
                cols.append(code_to_idx[code])
                data.append(1.0)  # 明确使用float类型
        
        # 创建稀疏矩阵，明确指定数据类型为float32
        feature_matrices[level] = csr_matrix((np.array(data, dtype=np.float32), 
                                             (np.array(rows), np.array(cols))), 
                                             shape=(n, len(code_to_idx)))
        
        # 转换为GPU稀疏矩阵
        feature_matrices_gpu[level] = cp_csr_matrix(feature_matrices[level])
        
        # # 保存编码字典以便后续使用
        # pd.DataFrame({'code': list(code_to_idx.keys()), 
        #               'index': list(code_to_idx.values())}).to_csv(
        #     f'{output_dir}/{level}_encoding.csv', index=False)
        
        # 清理内存
        del rows, cols, data
        gc.collect()
    
    return feature_matrices_gpu, n

# def process_batch_gpu(i, batch_size, n, feature_matrices_gpu, weights, output_dir, embeddings_array):
#     """处理单个批次的相似度计算"""
#     start_time = time.time()
#     print(f"处理批次 {i+1}/{(n + batch_size - 1) // batch_size}...")
#     start_i = i * batch_size
#     end_i = min((i + 1) * batch_size, n)
    
#     # 创建一个组合的相似度矩阵
#     combined_similarity = None
    
#     for level, weight in weights.items():
#         # 获取当前批次的特征矩阵（GPU版本）
#         batch_features = feature_matrices_gpu[level][start_i:end_i]
        
#         # 计算当前批次与所有专利的相似度（GPU加速）
#         level_similarity = batch_features.dot(feature_matrices_gpu[level].T)
        
#         # 应用权重
#         level_similarity = level_similarity.multiply(weight)
        
#         # 合并到总相似度矩阵
#         if combined_similarity is None:
#             combined_similarity = level_similarity
#         else:
#             combined_similarity = combined_similarity + level_similarity
    
#     # 对角线设为1（专利与自身的相似度）- 使用高效的矩阵操作替代循环
#     diagonal_indices = cp.arange(start_i, end_i)
#     row_indices = cp.arange(end_i - start_i)
#     combined_similarity[row_indices, diagonal_indices] = 1.0

#     # 计算摘要的embedding相似度
#     # 获取当前批次的摘要
#     # 计算摘要的embedding相似度（密集矩阵）
#     brief_similarity = cp.dot(embeddings_array[start_i:end_i], embeddings_array.T)

#     brief_similarity2 = cp.dot(embeddings_array[start_i:end_i], embeddings_array.T)
    
#     # 将combined_similarity从稀疏矩阵转换为密集矩阵，然后再相加
#     combined_similarity = combined_similarity.toarray()*0.4 + brief_similarity*0.4 + brief_similarity2*0.2 
    
    
#     # 将GPU结果转回CPU，并保存为稀疏矩阵
#     combined_similarity_cpu = combined_similarity.get()
#     # save_npz(f'{output_dir}/similarity_batch_{i}.npz', csr_matrix(combined_similarity_cpu))
    
#     end_time = time.time()
#     print(f"批次 {i+1} 处理完成，耗时: {end_time - start_time:.2f}秒")
#     return combined_similarity_cpu

# def process_batch_gpu(i, batch_size, n, feature_matrices_gpu, weights, output_dir, embeddings_array):
#     """处理单个批次的相似度计算并返回每行中值大于0.65的列索引"""
#     start_time = time.time()
#     print(f"处理批次 {i+1}/{(n + batch_size - 1) // batch_size}...")
#     start_i = i * batch_size
#     end_i = min((i + 1) * batch_size, n)
    
#     # 创建一个组合的相似度矩阵
#     combined_similarity = None
    
#     for level, weight in weights.items():
#         # 获取当前批次的特征矩阵（GPU版本）
#         batch_features = feature_matrices_gpu[level][start_i:end_i]
        
#         # 计算当前批次与所有专利的相似度（GPU加速）
#         level_similarity = batch_features.dot(feature_matrices_gpu[level].T)
        
#         # 应用权重
#         level_similarity = level_similarity.multiply(weight)
        
#         # 合并到总相似度矩阵
#         if combined_similarity is None:
#             combined_similarity = level_similarity
#         else:
#             combined_similarity = combined_similarity + level_similarity
    
#     # 对角线设为1（专利与自身的相似度）- 使用高效的矩阵操作替代循环
#     diagonal_indices = cp.arange(start_i, end_i)
#     row_indices = cp.arange(end_i - start_i)
#     combined_similarity[row_indices, diagonal_indices] = 1.0

#     # 计算摘要的embedding相似度
#     brief_similarity = cp.dot(embeddings_array[start_i:end_i], embeddings_array.T)
#     brief_similarity2 = cp.dot(embeddings_array[start_i:end_i], embeddings_array.T)
    
#     # 将combined_similarity从稀疏矩阵转换为密集矩阵，然后再相加
#     combined_similarity = combined_similarity.toarray()*0.4 + brief_similarity*0.4 + brief_similarity2*0.2 
    
#     # 在GPU上识别大于阈值的元素
#     similarity_mask = (combined_similarity > 0.8)

#     # 打印整个矩阵的最大值
#     print(combined_similarity.max())
    
#     # 将结果移到CPU上处理
#     similarity_mask_cpu = similarity_mask.get()
    
#     # 使用numpy的优化函数，为每一行查找满足条件的列索引
#     # 这比手动循环快得多
#     high_similarity_indices = [np.where(row_mask)[0].tolist() for row_mask in similarity_mask_cpu]
    
#     end_time = time.time()
#     print(f"批次 {i+1} 处理完成，耗时: {end_time - start_time:.2f}秒")
    
#     return high_similarity_indices

def process_batch_gpu(i, batch_size, n, feature_matrices_gpu, weights, brief_embeddings_array, title_embeddings_array, patent_ids):
    """处理单个批次的相似度计算并返回每行中值大于0.85的列索引及对应相似度值"""
    start_i = i * batch_size
    end_i = min((i + 1) * batch_size, n)
    
    # 创建一个组合的相似度矩阵
    combined_similarity = None
    
    for level, weight in weights.items():
        # 获取当前批次的特征矩阵（GPU版本）
        batch_features = feature_matrices_gpu[level][start_i:end_i]
        
        # 计算当前批次与所有专利的相似度（GPU加速）
        level_similarity = batch_features.dot(feature_matrices_gpu[level].T)
        
        # 应用权重
        level_similarity = level_similarity.multiply(weight)
        
        # 合并到总相似度矩阵
        if combined_similarity is None:
            combined_similarity = level_similarity
        else:
            combined_similarity = combined_similarity + level_similarity
    
    # 对角线设为1（专利与自身的相似度）- 使用高效的矩阵操作替代循环
    diagonal_indices = cp.arange(start_i, end_i)
    row_indices = cp.arange(end_i - start_i)
    combined_similarity[row_indices, diagonal_indices] = 1.0

    # 计算摘要的embedding相似度
    brief_similarity = cp.dot(brief_embeddings_array[start_i:end_i], brief_embeddings_array.T)
    title_similarity = cp.dot(title_embeddings_array[start_i:end_i], title_embeddings_array.T)
    
    # 将combined_similarity从稀疏矩阵转换为密集矩阵，然后再相加
    combined_similarity = combined_similarity.toarray()*0.4 + brief_similarity*0.4 + title_similarity*0.2
    
    # 在GPU上识别大于阈值的元素
    similarity_mask = (combined_similarity > 0.75)
    
    # 将相似度矩阵和掩码转移到CPU上处理
    similarity_cpu = combined_similarity.get()
    similarity_mask_cpu = similarity_mask.get()
    
    # 创建结果列表，同时包含索引和值
    high_similarity_results = []
    for row_idx in range(similarity_cpu.shape[0]):
        # 获取当前行中大于阈值的列索引
        col_indices = np.where(similarity_mask_cpu[row_idx])[0]
        # 获取这些位置的相似度值
        similarity_values = similarity_cpu[row_idx, col_indices]
        # 将索引和值打包在一起
        row_results = [(patent_ids[int(row_idx+start_i)], patent_ids[int(col_idx)], float(similarity_values[i])) 
                      for i, col_idx in enumerate(col_indices)]
        # 按相似度值降序排序
        row_results.sort(key=lambda x: x[2], reverse=True)
        high_similarity_results.append(row_results)
    
    return high_similarity_results

def merge_similarity_matrices(total_batches, output_dir):
    """合并所有批次的相似度矩阵"""
    print("合并相似度矩阵...")
    combined = None
    for i in range(total_batches):
        batch_sim = load_npz(f'{output_dir}/similarity_batch_{i}.npz')
        if combined is None:
            combined = batch_sim
        else:
            combined = vstack([combined, batch_sim])
    save_npz(f'{output_dir}/complete_similarity_matrix.npz', combined)
    print("合并完成，保存为 complete_similarity_matrix.npz")

# 加载专利的embedding npz
def load_patent_embeddings(file_name):
    data = cp.load(file_name)
    embeddings_array = data['embeddings']
    ids_array = data['ids']
    return embeddings_array, ids_array

def save_similiarity_results(similiarity_results, output_dir, ipc_code):
    # 将结果保存为csv文件
    pd.DataFrame(similiarity_results, columns=['patent_id', 'similar_patent_id', 'similarity_score']).to_csv(f'{output_dir}/similiarity_results_{ipc_code}.csv', index=False)

def main():
    # 设置GPU
    gpu_id = 0  # 修改这个值来选择不同的GPU
    setup_gpu(gpu_id)
    
    # 定义各层级权重
    weights = {
        # 'level1_code': 0.3,
        # 'level2_code': 0.2,
        'level3_code': 0.2,
        'level4_code': 0.3,
        'level5_code': 0.5
    }
    
    # 创建结果目录
    output_dir = 'similarity_results_gpu'
    os.makedirs(output_dir, exist_ok=True)
    
    # 加载数据
    merged_df = load_data('patent_data_D_cleaned.csv', 'ipc_categories_updated.csv')   ################
    
    # 创建索引映射表
    n = len(merged_df)
    if 'id' in merged_df.columns:
        index_mapping = pd.DataFrame({'row_idx': range(n), 'patent_id': merged_df['id']})
        index_mapping.to_csv(f'{output_dir}/index_mapping.csv', index=False)
    
    # 创建特征矩阵
    feature_matrices_gpu, n = create_feature_matrices(merged_df, weights, output_dir)

    # 加载专利的embedding npz
    embeddings_array, ids_array = load_patent_embeddings('patent_D_embeddings_0.npz')
    
    # 分批计算相似度矩阵
    print("开始计算分层相似度矩阵（GPU加速）...")
    batch_size = 2000  # GPU内存允许的情况下可以增加批次大小  ################
    total_batches = (n + batch_size - 1) // batch_size
    
    # 串行处理各个批次
    for i in range(total_batches):
        result = process_batch_gpu(i, batch_size, n, feature_matrices_gpu, weights, output_dir, embeddings_array)
        print(result)
    
    print(f"计算完成！所有结果已保存到 {output_dir} 目录下")
    
    # 合并结果
    merge_similarity_matrices(total_batches, output_dir)
    
    print("所有处理已完成，完整相似度矩阵已保存")


In [2]:
# 设置GPU
gpu_id = 1  # 修改这个值来选择不同的GPU
setup_gpu(gpu_id)

# 定义各层级权重
weights = {
    # 'level1_code': 0.3,
    # 'level2_code': 0.2,
    'level3_code': 0.2,
    'level4_code': 0.3,
    'level5_code': 0.5
}

# 创建结果目录
output_dir = 'similarity_results_gpu'
os.makedirs(output_dir, exist_ok=True)

# 加载数据
merged_df = load_data('patent_data_G06F_cleaned.csv', 'ipc_categories_updated.csv')   ################

# # 创建索引映射表
n = len(merged_df)
# if 'id' in merged_df.columns:
#     index_mapping = pd.DataFrame({'row_idx': range(n), 'patent_id': merged_df['id']})
#     index_mapping.to_csv(f'{output_dir}/index_mapping.csv', index=False)

# 创建特征矩阵
feature_matrices_gpu, n = create_feature_matrices(merged_df, weights, output_dir)

使用GPU 1: NVIDIA RTX 6000 Ada Generation
正在读取数据文件...
正在合并IPC类别信息...
进行高效预处理...
处理 level3_code...
处理 level4_code...
处理 level5_code...


In [2]:
import os
import glob

# 设置GPU
gpu_id = 0  # 修改这个值来选择不同的GPU
setup_gpu(gpu_id)

# 定义各层级权重
weights = {
    # 'level1_code': 0.3,
    # 'level2_code': 0.2,
    'level3_code': 0.2,
    'level4_code': 0.3,
    'level5_code': 0.5
}

patent_embedding_folder = "patent_embedding"
patent_data_folder = "patent_data"
output_dir = 'similarity_results_gpu'
# 获取所有npz文件路径
npz_files = glob.glob(os.path.join(patent_embedding_folder, "*.npz"))

IPC_3s = ['B05B']
# for file in npz_files:
#     # 从文件名中提取IPC_3代码
#     file_name = os.path.basename(file)
#     if "patent_brief_" in file_name and "_embeddings_0.npz" in file_name:
#         ipc_code = file_name.replace("patent_brief_", "").replace("_embeddings_0.npz", "")
#         IPC_3s.append(ipc_code)

# 进行保存的处理批次数量
save_batch = 1000

for ipc_code in IPC_3s:
    # 加载数据
    merged_df = load_data(os.path.join(patent_data_folder, 'patent_data_'+ipc_code+'_cleaned.csv'), 'ipc_categories_updated.csv')   ################
    # 创建索引映射表
    n = len(merged_df)
    # 创建特征矩阵
    feature_matrices_gpu, n = create_feature_matrices(merged_df, weights)
    # 加载专利的embedding npz
    brief_embeddings_array, brief_ids_array = load_patent_embeddings(os.path.join(patent_embedding_folder, 'patent_brief_'+ipc_code+'_embeddings_0.npz'))
    title_embeddings_array, title_ids_array = load_patent_embeddings(os.path.join(patent_embedding_folder, 'patent_title_'+ipc_code+'_embeddings_0.npz'))
    # 分批计算相似度矩阵
    print("开始计算分层相似度矩阵（GPU加速）...")
    batch_size = 500  # GPU内存允许的情况下可以增加批次大小  ################
    total_batches = (n + batch_size - 1) // batch_size

    similiarity_results = []
    # 串行处理各个批次
    for i in tqdm(range(total_batches), desc=f"Processing {ipc_code} batches"):
        result = process_batch_gpu(i, batch_size, n, feature_matrices_gpu, weights, brief_embeddings_array, title_embeddings_array, brief_ids_array.tolist())
        break
        for res in result:
            for r in res:
                similiarity_results.append(r)
        # 判断是否保存
        if i > 0 and i % save_batch == 0:
            save_similiarity_results(similiarity_results, output_dir, ipc_code+'_'+str(i))
            similiarity_results = []
    # 保存剩余结果
    # if len(similiarity_results) > 0:
    #     save_similiarity_results(similiarity_results, output_dir, ipc_code+'_'+str(total_batches))
    

使用GPU 0: NVIDIA RTX 6000 Ada Generation
正在读取数据文件...
正在合并IPC类别信息...
进行高效预处理...
处理 level3_code...
处理 level4_code...
处理 level5_code...
开始计算分层相似度矩阵（GPU加速）...


Processing B05B batches:   0%|          | 0/197 [00:03<?, ?it/s]


In [9]:
similiarity_results = []
for res in result:
    similiarity_results.extend(res)

In [10]:
similiarity_results[0]

(9939, 9939, 0.999981530372321)

In [18]:
42119107, 45650311, 45650139, 45648072, 45647246, 45646005
print(result[2][:100])
print(len(result[0]))

[(42119107, 45650311, 0.8973869331447076), (42119107, 45650139, 0.8774531701100955), (42119107, 45648072, 0.8965942072642065), (42119107, 45647246, 0.8911852054899554), (42119107, 45646005, 0.9130870927355772), (42119107, 45644043, 0.8908837377086457), (42119107, 45640337, 0.9082277726397934), (42119107, 45640156, 0.9149429084197891), (42119107, 45639553, 0.9157727607424022), (42119107, 45639529, 0.8870069415637317), (42119107, 45639181, 0.9050743914451231), (42119107, 45637493, 0.9080240550926789), (42119107, 45637371, 0.8675094708376492), (42119107, 45636625, 0.8876540889421564), (42119107, 45636344, 0.8492365436857086), (42119107, 45634963, 0.913491824808753), (42119107, 45634010, 0.8817330032981658), (42119107, 45633557, 0.9181368863355147), (42119107, 45633428, 0.9162752827365012), (42119107, 45631405, 0.907431073341104), (42119107, 45629251, 0.8768781468696716), (42119107, 45628257, 0.8833019038327337), (42119107, 45625564, 0.8820145785046862), (42119107, 45625175, 0.912170607381

In [25]:
# 使用embeddings_array的前100行与embeddings_array进行点积运算
result = cp.dot(embeddings_array[:100], embeddings_array.T)
s_time = time.time()
print(result.shape)
e_time = time.time()
print(f"点积运算耗时: {e_time - s_time:.2f}秒")


(100, 341518)
点积运算耗时: 0.00秒


In [26]:
# 使用embeddings_array的100行与embeddings_array进行点积运算
result = cp.dot(embeddings_array[100:200], embeddings_array.T)
s_time = time.time()
print(result.shape)
e_time = time.time()
print(f"点积运算耗时: {e_time - s_time:.2f}秒")

(100, 341518)
点积运算耗时: 0.00秒


In [22]:
merged_df.iloc[823846]

id                                                      34424301
title                                        行为控制方法、装置、电子设备及存储介质
brief          本发明实施例提供一种行为控制方法、装置、电子设备及存储介质；方法包括：在目标行为发生时，获取...
level1_code                                                    G
level2_code                                                  G06
level3_code                                                 G06F
level4_code                                            G06F21/00
level5_code                                            G06F21/52
Name: 823846, dtype: object

In [9]:
merged_df.iloc[0]

id                                                           138
title                                      应用软件连接数据库的方法、装置、设备及介质
brief          本发明实施例公开了一种应用软件连接数据库的方法、装置、设备及介质。该方法包括：响应于目标应用...
level1_code                                                    G
level2_code                                                  G06
level3_code                                                 G06F
level4_code                                            G06F21/00
level5_code                                            G06F21/52
Name: 0, dtype: object

In [ ]:
E01F,H05B,C22C,C09D,B29B,G08B,C12Q,F15B,H04M,B25H,A41D,D06F,B41F,A01C,F16C,G01D,A47C,E05B,F21V,C01B,E21D,A47G,E02B,B60L,G01F,B22D,H01Q,B62B,F16B,G02F,A61L,F25D,B66B,G05D,B28B,F04B,C12M,E01C,B66F,C23C,B23D,B05C,C07C,A23L,E01D,B01F,A63B,G01C,H02M,A47L,G09B,A61H,H04B,C04B,B23B,G01S,H04R,G09F,C08L,H01B,H01F,B07B,E04F,C07D,A61G,B62D,G06V,F16H,B60R,B05B

In [ ]:
E06B,B25J,B32B,B26D,B66C,E04H,F04D,C12N,A61F,A47B,H01H,H02G,F26B,B23P,G05B,H02B,A01K,E21B,B25B,F21S,F16M,A47J,E04G,E02D,H02K,F16L,G02B,G01M,B02C,G01B,H04N,F16K,G06T,E04B,B23Q,B65B,H02J,B01J,A61M,B65H,A01G,C02F,H04W,F24F,B08B,B21D

In [ ]:
H01R,B24B,H04L,A61K,H05K,H01M,G01R,G06Q,B23K,H01L,B65D,A61B,B29C,B65G,B01D,G01N,G06F

In [6]:
import os
import glob

patent_embedding_folder = "patent_embedding"
patent_data_folder = "patent_data"
output_dir = 'similarity_results_gpu'
# 获取所有npz文件路径
npz_files = glob.glob(os.path.join(patent_embedding_folder, "*.npz"))

IPC_3s = []
for file in npz_files:
    # 从文件名中提取IPC_3代码
    file_name = os.path.basename(file)
    if "patent_brief_" in file_name and "_embeddings_0.npz" in file_name:
        ipc_code = file_name.replace("patent_brief_", "").replace("_embeddings_0.npz", "")
        IPC_3s.append(ipc_code)

In [1]:
import os
import glob
import pandas as pd
from tqdm import tqdm

similarity_dir = 'similarity_results_gpu'
# 获取similarity_results_gpu目录下的所有parquet文件
similarity_parquet_files = glob.glob(os.path.join(similarity_dir, "*.parquet"))
# 获取similarity_results_gpu目录下的所有csv文件
similarity_csv_files = glob.glob(os.path.join(similarity_dir, "*.csv"))

# parquet中包含的IPC
parquet_ipc_list = []
for file in similarity_parquet_files:
    parquet_ipc_list.append(os.path.basename(file).split("similiarity_results_")[1].split("_")[0])

# 有效的csv文件
similarity_csv_files = [file for file in similarity_csv_files if os.path.basename(file).split("similiarity_results_")[1].split("_")[0] not in parquet_ipc_list]


In [2]:
# csv中包含的IPC
ipc_list = []

for file in similarity_parquet_files: 
    ipc_list.append(os.path.basename(file).split("similiarity_results_")[1].split("_")[0])
for file in similarity_csv_files: 
    ipc_list.append(os.path.basename(file).split("similiarity_results_")[1].split("_")[0])

assert len(set(ipc_list)) == 644

In [4]:
# parquet结果列表
parquet_result_list = []
# 读取parquet文件为dataframe
for file in tqdm(similarity_parquet_files, desc="Processing parquet files"):
    tqdm.write(f"正在处理文件: {file}")
    df = pd.read_parquet(file)
    df = df[df['patent_id'] != df['similar_patent_id']]
    # 不使用 apply 的替代方案
    # 首先按 patent_id 和 similarity_score 排序
    df_sorted = df.sort_values(['patent_id', 'similarity_score'], ascending=[True, False])
    # 然后使用 groupby 和 head 获取每组前 100 条
    result = df_sorted.groupby('patent_id').head(100).reset_index(drop=True)
    parquet_result_list.append(result)


Processing parquet files:   0%|          | 0/275 [00:00<?, ?it/s]

正在处理文件: similarity_results_gpu/similiarity_results_B62K_25.parquet


Processing parquet files:   0%|          | 1/275 [00:01<06:57,  1.52s/it]

正在处理文件: similarity_results_gpu/similiarity_results_C21D_40.parquet


Processing parquet files:   1%|          | 2/275 [00:06<15:52,  3.49s/it]

...

Processing parquet files:  99%|█████████▉| 273/275 [4:57:43<01:35, 47.97s/it]

正在处理文件: similarity_results_gpu/similiarity_results_B64D_27.parquet


Processing parquet files: 100%|█████████▉| 274/275 [4:57:54<00:36, 36.93s/it]

正在处理文件: similarity_results_gpu/similiarity_results_B60T_25.parquet


Processing parquet files: 100%|██████████| 275/275 [4:57:57<00:00, 65.01s/it]


In [6]:
parquet_df = pd.concat(parquet_result_list, ignore_index=True)

In [7]:
parquet_df.to_csv('parquet_result.csv', index=False)

In [17]:
parquet_csv_list = []
# 读取parquet文件为dataframe
for file in tqdm(similarity_csv_files, desc="Processing csv files"):
    tqdm.write(f"正在处理文件: {file}")
    df = pd.read_csv(file)
    df = df[df['patent_id'] != df['similar_patent_id']]
    # 不使用 apply 的替代方案
    # 首先按 patent_id 和 similarity_score 排序
    df_sorted = df.sort_values(['patent_id', 'similarity_score'], ascending=[True, False])
    # 然后使用 groupby 和 head 获取每组前 100 条
    result = df_sorted.groupby('patent_id').head(100).reset_index(drop=True)
    parquet_csv_list.append(result)

In [8]:
parquet_df.shape

(1468672994, 3)

In [9]:
99642574 + 1468672994

1568315568